In [1]:
# CELL 1: Imports

import pandas as pd
import numpy as np
from pathlib import Path

df = pd.read_csv(
    "../data/processed/logistics_cleaned.csv"
)

print("Dataset loaded:", df.shape)

Dataset loaded: (180519, 40)


In [2]:
# CELL 2: Delivery duration gap

df["delivery_duration_gap"] = (
    df["days_for_shipping_real"] -
    df["days_for_shipment_scheduled"]
)

display(df[[
    "days_for_shipping_real",
    "days_for_shipment_scheduled",
    "delivery_duration_gap"
]].head())

,days_for_shipping_real,days_for_shipment_scheduled,delivery_duration_gap
0,6,4,2
1,4,4,0
2,4,4,0
3,6,4,2
4,6,4,2


In [3]:
# CELL 3: Delivery performance classification

def classify_delivery(gap):
    if gap > 0:
        return "Delayed"
    elif gap == 0:
        return "On-time"
    else:
        return "Early"

df["duration_performance"] = df[
    "delivery_duration_gap"
].apply(classify_delivery)

display(df["duration_performance"].value_counts())

duration_performance
Delayed    103400
Early       43366
On-time     33753
Name: count, dtype: int64

In [4]:
# CELL 4: Delay severity

df["delay_severity"] = np.select(
    [
        df["delivery_duration_gap"] > 3,
        df["delivery_duration_gap"] > 1,
        df["delivery_duration_gap"] > 0
    ],
    [
        "High",
        "Medium",
        "Low"
    ],
    default="No Delay"
)

display(df["delay_severity"].value_counts())

delay_severity
No Delay    77119
Low         60647
Medium      35770
High         6983
Name: count, dtype: int64

In [5]:
# CELL 5: Additional operational features

df["shipping_efficiency"] = (
    df["days_for_shipment_scheduled"] /
    df["days_for_shipping_real"].replace(0, np.nan)
)

df["shipping_efficiency"] = df[
    "shipping_efficiency"
].replace([np.inf, -np.inf], np.nan)

df["shipping_efficiency"] = df[
    "shipping_efficiency"
].fillna(0)

display(df.head())

,type,days_for_shipping_real,days_for_shipment_scheduled,benefit_per_order,sales_per_customer,delivery_status,late_delivery_risk,category_id,category_name,customer_city,...,order_region,order_state,order_status,product_name,product_price,shipping_mode,delivery_duration_gap,duration_performance,delay_severity,shipping_efficiency
0,DEBIT,6,4,159.69,472.45,Late delivery,1,9,Cardio Equipment,Brownsville,...,South Asia,Maharashtra,COMPLETE,Nike Men's Free 5.0+ Running Shoe,99.99,Standard Class,2,Delayed,Medium,0.666667
1,DEBIT,4,4,48.71,167.96,Shipping on time,0,29,Shop By Sport,Littleton,...,Central America,Cortés,ON_HOLD,Under Armour Girls' Toddler Spine Surge Runni,39.99,Standard Class,0,On-time,No Delay,1.000000
2,DEBIT,4,4,87.36,181.99,Shipping on time,0,48,Water Sports,Littleton,...,Central America,Cortés,ON_HOLD,Pelican Sunstream 100 Kayak,199.99,Standard Class,0,On-time,No Delay,1.000000
3,DEBIT,6,4,-41.89,175.99,Late delivery,1,48,Water Sports,Littleton,...,East of USA,Nueva York,COMPLETE,Pelican Sunstream 100 Kayak,199.99,Standard Class,2,Delayed,Medium,0.666667
4,DEBIT,6,4,10.00,40.00,Late delivery,1,24,Women's Apparel,Littleton,...,East of USA,Nueva York,COMPLETE,Nike Men's Dri-FIT Victory Golf Polo,50.00,Standard Class,2,Delayed,Medium,0.666667


In [6]:
# CELL 6: Save feature-engineered dataset

OUTPUT_DIR = Path("../data/processed")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

df.to_csv(
    OUTPUT_DIR / "logistics_features.csv",
    index=False
)

print("Feature engineering completed!")

Feature engineering completed!
